# Likelihood Ratio Attack
This notebook is an implementation of the Likelihood Ratio Attack with phi scores using scaled logits and other variations

# Sampling the public data

In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset
from torchvision.models import resnet18
import torchvision.transforms as transforms
from pathlib import Path
import numpy as np
from torch.utils.data import Subset
import torch.nn as nn

In [ ]:
#Config
PUB_PATH = '/content/pub.pt'
NUM_SHADOWS= 8

In [ ]:
# dataset classes
class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids = []
        self.imgs = []
        self.labels = []
        self.transform = transform

    def __getitem__(self, index):
        id_ = self.ids[index]
        img = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        label = self.labels[index]
        return id_, img, label

    def __len__(self):
        return len(self.ids)


class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []

    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]

In [ ]:
# load datasets
print("Loading datasets...")
pub_ds = torch.load(PUB_PATH, weights_only=False)
print(f"Public dataset: {len(pub_ds)} samples")

Loading datasets...
Public dataset: 14000 samples


In [ ]:
# normalization (same as training)
MEAN = [0.7406, 0.5331, 0.7059]
STD = [0.1491, 0.1864, 0.1301]

transform = transforms.Compose([
    transforms.Resize(32),
    transforms.Normalize(mean=MEAN, std=STD),
])

pub_ds.transform = transform

# Dataset Sampling for Shadow models

In [ ]:
def sample_shadow_dataset(public_dataset, seed, fraction=0.5):

    np.random.seed(seed)
    n = len(public_dataset)
    indices = np.random.permutation(n)
    train_size = int(fraction * n)
    train_idx = indices[:train_size]
    holdout_idx = indices[train_size:]
    return Subset(public_dataset, train_idx), Subset(public_dataset, holdout_idx), train_idx, holdout_idx

# Shadow Model (ResNet18)

In [ ]:
def build_model():

    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 9)
    return model

In [ ]:
#training loop
def train(model, train_loader, device, epochs=100):
    model.to(device)
    optimiser= torch.optim.SGD(
        model.parameters(),
        lr=0.01,
        momentum=0.9,
        weight_decay=5e-4,
    )
    for epoch in range(epochs):
      model.train()
      for _,x,y,_ in train_loader:
        x,y = x.to(device), y.to(device)
        optimiser.zero_grad()
        output = model(x)
        criterion= nn.CrossEntropyLoss()
        loss = criterion(output, y)
        loss.backward()
        optimiser.step()
    return model

# Compute Scores using Scaled logits

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_phi(model, loader, device, eps=1e-8):
    """
    Computes LiRA phi scores using logit-scaled confidence:
        phi = log(p_y/(1 - p_y))
    """
    scores = {}
    model.eval()

    with torch.no_grad():
        for ids, x, y, _ in loader:
            x, y = x.to(device), y.to(device)

            # Raw logits
            logits = model(x)

            # Convert logits to probabilities
            probs = F.softmax(logits, dim=1)

            # True class probability p_y
            p_y = probs[torch.arange(len(y)), y]

            # Logit-scaled phi score
            phi = torch.log(p_y + eps) - torch.log(1 - p_y + eps)

            # Store scores
            phi = phi.cpu().numpy()

            for i, sample_id in enumerate(ids):
                scores[int(sample_id)] = phi[i]

    return scores


In [ ]:
from torch.utils.data.dataloader import DataLoader
# Compute in and out scores
all_in_scores={}
all_out_scores={}

for shadow_idx in range(NUM_SHADOWS):
  # using different seed value to ensure different random sampling config for each shadow model
  train_ds, holdout_ds, train_idx, holdout_idx= sample_shadow_dataset(pub_ds, seed= 42+ shadow_idx)
  train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
  holdout_loader = DataLoader(holdout_ds, batch_size=128, shuffle=False)

  shadow_model = build_model()
  shadow_model = train(shadow_model, train_loader, device='cuda')

  #compute phi scores
  train_scores= compute_phi(shadow_model, train_loader, device='cuda')
  holdout_scores= compute_phi(shadow_model, holdout_loader, device='cuda')
  #save the in and out scores
  for idx, score in train_scores.items():
    all_in_scores.setdefault(idx, []).append(score)
  for idx, score in holdout_scores.items():
    all_out_scores.setdefault(idx, []).append(score)

  print(f'Shadow model {shadow_idx} done')

print(f'in_scores={train_scores}')
print(f'out_scores={holdout_scores}')

Shadow model 0 done
Shadow model 1 done
Shadow model 2 done
Shadow model 3 done
in_scores={1499: np.float32(12.505692), 63233: np.float32(9.783056), 18647: np.float32(10.211976), 79027: np.float32(10.911381), 35603: np.float32(9.388248), 91966: np.float32(12.27667), 104258: np.float32(9.600048), 5980: np.float32(10.028611), 57437: np.float32(7.99434), 75130: np.float32(11.205438), 942: np.float32(7.304236), 90345: np.float32(8.648115), 32621: np.float32(8.667676), 67092: np.float32(8.095457), 78800: np.float32(11.707055), 75373: np.float32(14.816195), 65619: np.float32(10.347329), 92029: np.float32(11.546891), 106019: np.float32(9.221965), 80537: np.float32(12.993539), 45559: np.float32(9.46367), 69650: np.float32(10.104369), 42031: np.float32(13.047362), 70851: np.float32(11.07419), 69552: np.float32(12.681065), 96799: np.float32(9.199305), 6124: np.float32(13.297352), 92458: np.float32(12.00891), 93653: np.float32(7.760075), 66818: np.float32(10.146033), 81907: np.float32(10.40469), 

# Fit to Gaussian Distribution

In [ ]:
member_scores = []
nonmember_scores = []

for scores in all_in_scores.values():
    member_scores.extend(scores)

for scores in all_out_scores.values():
    nonmember_scores.extend(scores)

mu_in = np.mean(member_scores)
sigma_in = max(np.std(member_scores), 1e-6)
mu_out = np.mean(nonmember_scores)
sigma_out = max(np.std(nonmember_scores), 1e-6)

# save the statistics
torch.save({
    "mu_in": mu_in,
    "sigma_in": sigma_in,
    "mu_out": mu_out,
    "sigma_out": sigma_out
}, "lira__logits_stats.pt")

In [ ]:
#testing TPR@5%FPR
def tpr_at_fpr(member_scores, nonmember_scores, fpr=0.05):

    threshold = np.percentile(nonmember_scores, 100 * (1 - fpr))

    return np.mean(np.array(member_scores) > threshold)

In [ ]:
print(f'TPR@5%FPR : {tpr_at_fpr(member_scores, nonmember_scores, fpr=0.05)}')

TPR@5%FPR : 0.072
